# Three-Parameter Cubic EOS + Chao-Seader (M7.3, planned v0.5.0)

> ⚠️ **DRAFT — M7.3 planned for `v0.5.0`.** The functionality this notebook will exercise has **not yet been implemented** in the engine. Today (v0.3.0) the calls in the "behavior today" section below raise `NotImplementedError` or panic with an `M7.x deferred` marker — that's intentional, not a bug. For what does work right now, see [`02_pure_component.ipynb`](02_pure_component.ipynb).

Ports the **Pascal-origin three-parameter EOS** — Schmidt-Wenzel, Patel-Teja, Patel-Teja USB — plus the Chao-Seader liquid fugacity correlation. All from `legacy/pascal/TERMOII.PAS`, Ref (4) Da Silva & Báez (1989).


> 💾 **Hub sandbox notice — only applies if you're running this notebook on the hosted VLE JupyterLab.** If the VLE developers gave you a URL to a shared JupyterLab environment, that environment is an *educational sandbox*: edits you make to this notebook won't survive a container restart, the bundled `vle-thermo` version may lag PyPI, and any `pip install` you run inside this container is ephemeral (it vanishes when your session is culled). For real work, install `vle-thermo` in your own Jupyter environment with `pip install vle-thermo` and run the notebook there — see the [project README](https://github.com/miguelju/vle/blob/main/README.md). **If you opened this notebook in your own Jupyter, you can ignore this notice.**

## Setup (optional)

The cell below is **commented out by default**. Uncomment it if you want to use the latest `vle-thermo` released on PyPI instead of whatever version is currently installed in your kernel — this matters most for *this* notebook because the feature being demonstrated is **planned for a future release**, and you may already be on it by the time you read this.

In [1]:
# Optional: pull the latest vle-thermo from PyPI.
# Uncomment if you want the newest released version instead of
# whatever is currently in your kernel. On the hosted hub this
# install is ephemeral — it vanishes when your session is culled.
# %pip install --upgrade vle-thermo

## Why this is coming

Two-parameter cubic EOS struggle with polar molecules (water, alcohols) and very asymmetric mixtures (light gas + heavy hydrocarbon). The Pascal program shipped three additions to handle those: Schmidt-Wenzel adds an acentric-factor-dependent covolume `β`, Patel-Teja uses a fitted Z_c correlation as the third parameter, and Chao-Seader provides a semi-empirical liquid-fugacity correlation with separate coefficient sets for hydrogen, methane, and "normal" compounds. Each addition is small and well-bounded; together they let v0.5.0 cover the Chapter IV validation cases that 2-parameter EOS alone cannot.

## Planned scope (M7.3 → `v0.5.0`)

| Component | Source | Notes |
|---|---|---|
| Schmidt-Wenzel | TERMOII.PAS | Beta(ω) ⇒ k₁(ω), k₂(ω) per-component; special C-parameter mixing |
| Patel-Teja (`PatelT`) | TERMOII.PAS | Fitted Z_c, c = OmC·R·T_c/P_c; mole-fraction C-mixing |
| Patel-Teja USB (`PatelTUSB`) | TERMOII.PAS | Same form, √B-weighted C-mixing |
| Chao-Seader liquid fugacity | TERMOII.PAS | 10+ params; H₂ / methane special cases |

All four entries already have enum slots in `engine/src/eos.rs::CubicEos` (`SchmidtWenzel`, `PatelTeja`, `PatelTejaUSB`) and `LiquidModel::ChaoSeader`. M7.3 fills in the actual EOS-specific code paths and a sub-section in the C-mixing rule module.

## Behavior today (v0.3.0)

The cell below calls the planned feature **through the existing engine binding**. Today it raises the deferred-stub error so you can see exactly what M7.3 will eventually replace. Once that milestone ships in `v0.5.0`, the same cell will produce real numbers — and this banner will go away.

In [2]:
from vle._engine import CubicEos, eos_z_factor

# Schmidt-Wenzel is M7.3-deferred — should raise NotImplementedError.
try:
    eos_z_factor(CubicEos.SchmidtWenzel, 300.0, 1000.0,
                 190.564, 4599.0, 0.0115, 'vapor')
    print('ERROR: expected NotImplementedError')
except NotImplementedError as exc:
    print(f'NotImplementedError: {exc}')

NotImplementedError: EOS variant SchmidtWenzel not yet ported — see M7 sub-milestones


## References

- **ROADMAP.md** — the live status of this sub-milestone. [`ROADMAP.md`](https://github.com/miguelju/vle/blob/main/ROADMAP.md)
- **MODERNIZATION_PLAN.md** — phase-level technical scope. [`MODERNIZATION_PLAN.md`](https://github.com/miguelju/vle/blob/main/MODERNIZATION_PLAN.md)
- **v0.3.0 functional notebook** — what works today. [`02_pure_component.ipynb`](02_pure_component.ipynb)